<a href="https://colab.research.google.com/github/nattynatasha-gif/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duka Electronics Sales Exploration
Half-year review (January–June 2026)

## Executive Summary
- Nairobi CBD generated the highest revenue at **KES 25,855,067**, about 28% of the half-year total of KES 93.2 million.
- Branch rankings by revenue and by number of orders are identical: Nairobi CBD (277 orders) leads and Nakuru (101 orders) is last.
- Top three products by revenue are iPhone 13 (KES 13.5 m), Lenovo IdeaPad 3 (KES 11.6 m) and HP Pavilion 15 (KES 10.8 m).
- Weekend orders (279) have a higher average order value (KES 96,590) than weekday orders (KES 91,896).
- February was the best month (KES 16.8 m) and May the worst (KES 14.1 m), a difference of 19.2%.


## First-five-minutes inspection
- Shape: 1,000 rows × 8 columns
- Types: order_date is datetime, quantity & unit_price are integers, rest are strings
- Missing values: none
- Date range: 2026-01-01 to 2026-06-30
- Key categories: 6 branches, 14 products, 2 channels (In-store 692 / Online 308)

In [2]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("Shape:", df.shape)
print(df.dtypes)
print(df.isna().sum())
print("Date range:", df["order_date"].min().date(), "→", df["order_date"].max().date())
print(df["branch"].value_counts())
print(df["channel"].value_counts())

Shape: (1000, 8)
order_id              object
order_date    datetime64[ns]
branch                object
city                  object
product               object
quantity               int64
unit_price             int64
channel               object
dtype: object
order_id      0
order_date    0
branch        0
city          0
product       0
quantity      0
unit_price    0
channel       0
dtype: int64
Date range: 2026-01-01 → 2026-06-30
branch
Nairobi CBD    277
Westlands      211
Mombasa        162
Kisumu         134
Eldoret        115
Nakuru         101
Name: count, dtype: int64
channel
In-store    692
Online      308
Name: count, dtype: int64


In [3]:
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.month
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")
df[["order_id", "revenue", "month", "weekday", "is_weekend", "price_band"]].head()

,order_id,revenue,month,weekday,is_weekend,price_band
0,ORD00001,143652,6,Thursday,False,Premium
1,ORD00002,11787,2,Monday,False,Value
2,ORD00003,196148,1,Monday,False,Premium
3,ORD00004,89080,4,Sunday,True,Value
4,ORD00005,130221,3,Sunday,True,Premium


# Task 3 – Rank branches

In [4]:
rev_rank = df.groupby("branch")["revenue"].sum().sort_values(ascending=False)
ord_rank = df.groupby("branch").size().sort_values(ascending=False)
print("Revenue ranking:\n", rev_rank)
print("\nOrder ranking:\n", ord_rank)
print("\nSame order?", list(rev_rank.index) == list(ord_rank.index))

Revenue ranking:
 branch
Nairobi CBD    25855067
Westlands      18698885
Mombasa        15268267
Kisumu         12781867
Eldoret        10771938
Nakuru          9829511
Name: revenue, dtype: int64

Order ranking:
 branch
Nairobi CBD    277
Westlands      211
Mombasa        162
Kisumu         134
Eldoret        115
Nakuru         101
dtype: int64

Same order? True


**Answer:** Yes, the rankings are the same. Nairobi CBD leads both lists. Average order value is similar across branches, so the branch with the most orders also has the highest revenue.

# Task 4 – Top products

In [5]:
print("Top 3 by revenue:\n", df.groupby("product")["revenue"].sum().nlargest(3))
print("\nTop 3 by units:\n", df.groupby("product")["quantity"].sum().nlargest(3))

Top 3 by revenue:
 product
iPhone 13           13541182
Lenovo IdeaPad 3    11618748
HP Pavilion 15      10827881
Name: revenue, dtype: int64

Top 3 by units:
 product
Canon EOS 2000D        219
LG 43-inch Smart TV    216
Lenovo IdeaPad 3       212
Name: quantity, dtype: int64


**Answer:** Revenue leaders are high-ticket items (iPhone, laptops). Unit leaders are mid-priced items that sell more pieces. Only Lenovo IdeaPad 3 appears in both lists.

# Task 5 – Weekend vs weekday

In [6]:
print(df.groupby("is_weekend").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_order_revenue=("revenue", "mean")
).round(0))

            orders  total_revenue  avg_order_revenue
is_weekend                                          
False          721       66256946            91896.0
True           279       26948589            96590.0


**Answer:** Weekdays have more orders and higher total revenue. Weekends have a higher average order value (KES 96,590 vs 91,896).

# Task 6 – Best and worst month

In [7]:
mrev = df.groupby("month")["revenue"].sum()
print(mrev)
best, worst = mrev.idxmax(), mrev.idxmin()
pct = (mrev[best] - mrev[worst]) / mrev[worst] * 100
print(f"Best: {best} (KES {mrev[best]:,.0f})")
print(f"Worst: {worst} (KES {mrev[worst]:,.0f})")
print(f"Difference: {pct:.1f}%")

month
1    16184935
2    16807197
3    16616894
4    15307010
5    14099397
6    14190102
Name: revenue, dtype: int64
Best: 2 (KES 16,807,197)
Worst: 5 (KES 14,099,397)
Difference: 19.2%


**Answer:** February is the best month and May is the worst. The percentage difference is 19.2%

# Task 7 – Premium Online Nairobi & Mombasa

In [8]:
prem = df[(df["price_band"] == "Premium") & (df["channel"] == "Online") & (df["city"].isin(["Nairobi", "Mombasa"]))]
print("Count:", len(prem))
print("Total revenue: KES", f"{prem['revenue'].sum():,.0f}")
os.makedirs("outputs", exist_ok=True)
prem.to_csv("outputs/premium_online_coast_capital.csv", index=False)
print("Saved to outputs/premium_online_coast_capital.csv")

Count: 111
Total revenue: KES 13,471,660
Saved to outputs/premium_online_coast_capital.csv


**Answer:** There are 111 Premium Online orders from Nairobi and Mombasa with total revenue of KES 13,471,660.

In [9]:
os.makedirs("excel", exist_ok=True)
branch_rev = df.groupby("branch")["revenue"].sum().reset_index()
branch_rev.columns = ["Branch", "Total_Revenue"]
branch_rev = branch_rev.sort_values("Total_Revenue", ascending=False)
region_map = {"Nairobi CBD": "Capital", "Westlands": "Capital", "Mombasa": "Coast",
              "Kisumu": "Western", "Nakuru": "Rift Valley", "Eldoret": "Rift Valley"}
branch_rev["Region"] = branch_rev["Branch"].map(region_map)
pivot = df.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum", fill_value=0)
pivot["Total"] = pivot.sum(axis=1)
pivot = pivot.reset_index()
with pd.ExcelWriter("excel/duka_branch_summary.xlsx", engine="openpyxl") as writer:
    branch_rev.to_excel(writer, sheet_name="Branch_Revenue_SUMIFS", index=False)
    pd.DataFrame(list(region_map.items()), columns=["Branch", "Region"]).to_excel(writer, sheet_name="Region_Lookup", index=False)
    pivot.to_excel(writer, sheet_name="Pivot_Branch_Channel", index=False)
print("Excel saved. Totals match:", df["revenue"].sum() == pivot["Total"].sum())

Excel saved. Totals match: True
